<a href="https://colab.research.google.com/github/ankitarchoudhary/IBM-Data-Science-Capstone/blob/main/Phase4_EDA_SQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 4: Exploratory Data Analysis (EDA) with SQL


This phase answers specific questions about the dataset using SQL queries instead of pandas code.
It uses SQLite, a lightweight database built into Python (no separate database server or account
needed), to load the CSV into a queryable table.


## Load the Data into a SQL Database

In [1]:
import pandas as pd
import sqlite3

df = pd.read_csv('spacex_wrangled.csv')

conn = sqlite3.connect(':memory:')
df.to_sql('spacex', conn, index=False, if_exists='replace')

print("Table 'spacex' created with", len(df), "rows.")

Table 'spacex' created with 90 rows.


**Explanation:**
- `sqlite3` is Python's built-in library for working with SQLite, a lightweight database that
  does not require installing or connecting to a separate server.
- `sqlite3.connect(':memory:')` creates a temporary database that exists only in memory for this
  session (nothing is saved to disk).
- `df.to_sql('spacex', conn, index=False, if_exists='replace')` copies the DataFrame into a SQL
  table named `spacex` inside that database, so it can be queried with SQL statements. `index=False`
  means the row numbers are not saved as an extra column, and `if_exists='replace'` recreates the
  table if this cell is run more than once.

## Query 1: All Unique Launch Sites

In [2]:
query = "SELECT DISTINCT LaunchSite FROM spacex;"
pd.read_sql_query(query, conn)

,LaunchSite
0,CCAFS SLC 40
1,VAFB SLC 4E
2,KSC LC 39A


**Explanation:**
- `pd.read_sql_query(query, conn)` runs a SQL query against the `spacex` table and returns the
  result as a DataFrame.
- `SELECT DISTINCT LaunchSite` returns each unique value in the `LaunchSite` column exactly once,
  which is the SQL equivalent of pandas' `.unique()`.

## Query 2: Records Where Launch Site Begins with 'CCA'

In [3]:
query = """
SELECT FlightNumber, Date, BoosterVersion, LaunchSite, PayloadMass, Orbit, Outcome
FROM spacex
WHERE LaunchSite LIKE 'CCA%'
LIMIT 5;
"""
pd.read_sql_query(query, conn)

,FlightNumber,Date,BoosterVersion,LaunchSite,PayloadMass,Orbit,Outcome
0,1,2010-06-04,Falcon 9,CCAFS SLC 40,6104.959412,LEO,None None
1,2,2012-05-22,Falcon 9,CCAFS SLC 40,525.000000,LEO,None None
2,3,2013-03-01,Falcon 9,CCAFS SLC 40,677.000000,ISS,None None
3,5,2013-12-03,Falcon 9,CCAFS SLC 40,3170.000000,GTO,None None
4,6,2014-01-06,Falcon 9,CCAFS SLC 40,3325.000000,GTO,None None


**Explanation:**
- `LIKE 'CCA%'` matches any `LaunchSite` value that starts with the text "CCA" - the `%` is a
  wildcard that matches any characters after it.
- `LIMIT 5` restricts the result to the first 5 matching rows.

## Query 3: Total Payload Mass Across All Launches

In [4]:
query = "SELECT SUM(PayloadMass) AS total_payload_mass FROM spacex;"
pd.read_sql_query(query, conn)

,total_payload_mass
0,549446.347059


**Explanation:** `SUM(PayloadMass)` adds up the `PayloadMass` value across every row in the
table. `AS total_payload_mass` renames the result column for clarity.

## Query 4: Average Payload Mass by Orbit

In [5]:
query = """
SELECT Orbit, AVG(PayloadMass) AS avg_payload_mass, COUNT(*) AS num_launches
FROM spacex
GROUP BY Orbit
ORDER BY avg_payload_mass DESC;
"""
pd.read_sql_query(query, conn)

,Orbit,avg_payload_mass,num_launches
0,VLEO,15315.714286,14
1,PO,7583.666667,9
2,SO,6104.959412,1
3,GEO,6104.959412,1
4,GTO,5011.994444,27
5,MEO,3987.000000,3
6,LEO,3882.839748,7
7,ISS,3279.938095,21
8,SSO,2060.000000,5
9,ES-L1,570.000000,1


**Explanation:**
- `GROUP BY Orbit` splits the table into groups, one per distinct orbit type.
- `AVG(PayloadMass)` computes the average payload mass within each group, and `COUNT(*)` counts
  how many launches are in each group.
- `ORDER BY avg_payload_mass DESC` sorts the results from highest to lowest average payload mass.

## Query 5: Date of the First Successful Landing

In [6]:
query = "SELECT MIN(Date) AS first_successful_landing FROM spacex WHERE Class = 1;"
pd.read_sql_query(query, conn)

,first_successful_landing
0,2014-04-18


**Explanation:**
- `WHERE Class = 1` filters the table to only rows representing a successful landing.
- `MIN(Date)` finds the earliest date among those filtered rows, since dates sort chronologically
  as text in the `YYYY-MM-DD` format used here.

## Query 6: Successful Landings with Payload Mass Between 4000 and 6000 kg

In [7]:
query = """
SELECT Serial, BoosterVersion, LaunchSite, PayloadMass
FROM spacex
WHERE Class = 1 AND PayloadMass BETWEEN 4000 AND 6000;
"""
pd.read_sql_query(query, conn)

,Serial,BoosterVersion,LaunchSite,PayloadMass
0,B1022,Falcon 9,CCAFS SLC 40,4696.0
1,B1026,Falcon 9,CCAFS SLC 40,4600.0
2,B1021,Falcon 9,KSC LC 39A,5300.0
3,B1040,Falcon 9,KSC LC 39A,4990.0
4,B1031,Falcon 9,KSC LC 39A,5200.0
5,B1032,Falcon 9,CCAFS SLC 40,4230.0
6,B1046,Falcon 9,CCAFS SLC 40,5800.0
7,B1046,Falcon 9,VAFB SLC 4E,4000.0
8,B1059,Falcon 9,CCAFS SLC 40,5000.0


**Explanation:**
- `WHERE Class = 1 AND PayloadMass BETWEEN 4000 AND 6000` combines two conditions: the landing
  must have succeeded, and the payload mass must fall within the given range (inclusive of both
  4000 and 6000).

## Query 7: Total Successful vs. Unsuccessful Landings

In [8]:
query = """
SELECT
    CASE WHEN Class = 1 THEN 'Successful' ELSE 'Unsuccessful' END AS landing_result,
    COUNT(*) AS total_count
FROM spacex
GROUP BY Class;
"""
pd.read_sql_query(query, conn)

,landing_result,total_count
0,Unsuccessful,30
1,Successful,60


**Explanation:**
- `CASE WHEN Class = 1 THEN 'Successful' ELSE 'Unsuccessful' END` converts the numeric `Class`
  value into a readable label, purely for display purposes.
- `GROUP BY Class` then counts how many rows fall into each of the two outcome groups.

## Query 8: Booster(s) That Carried the Maximum Payload Mass

In [9]:
query = """
SELECT Serial, BoosterVersion, LaunchSite, PayloadMass
FROM spacex
WHERE PayloadMass = (SELECT MAX(PayloadMass) FROM spacex);
"""
pd.read_sql_query(query, conn)

,Serial,BoosterVersion,LaunchSite,PayloadMass
0,B1048,Falcon 9,CCAFS SLC 40,15600.0
1,B1051,Falcon 9,CCAFS SLC 40,15600.0
2,B1048,Falcon 9,KSC LC 39A,15600.0


**Explanation:**
- The inner query `(SELECT MAX(PayloadMass) FROM spacex)` first finds the single highest payload
  mass value in the entire table.
- The outer query then finds every row whose `PayloadMass` matches that maximum value - this
  correctly returns multiple boosters if more than one launch shares the same maximum payload.

## Query 9: Unsuccessful Landings in 2020

In [10]:
query = """
SELECT Date, Serial, LaunchSite, Outcome
FROM spacex
WHERE Class = 0 AND Date LIKE '2020%';
"""
pd.read_sql_query(query, conn)

,Date,Serial,LaunchSite,Outcome
0,2020-01-19,B1046,KSC LC 39A,None None
1,2020-02-17,B1056,CCAFS SLC 40,False ASDS
2,2020-03-18,B1048,KSC LC 39A,False ASDS


**Explanation:**
- `Date LIKE '2020%'` matches any date starting with "2020" (since dates are stored in
  `YYYY-MM-DD` format, this reliably filters to that year).
- Combined with `Class = 0`, this returns only the unsuccessful landings that happened in 2020.

## Query 10: Ranking Landing Outcomes Between Two Dates

In [11]:
query = """
SELECT Outcome, COUNT(*) AS count_outcomes
FROM spacex
WHERE Date BETWEEN '2015-01-01' AND '2018-01-01'
GROUP BY Outcome
ORDER BY count_outcomes DESC;
"""
pd.read_sql_query(query, conn)

,Outcome,count_outcomes
0,True ASDS,12
1,True RTLS,8
2,None None,4
3,False ASDS,4
4,True Ocean,2
5,None ASDS,2


**Explanation:**
- `WHERE Date BETWEEN '2015-01-01' AND '2018-01-01'` filters to launches within that date range.
- `GROUP BY Outcome` groups the filtered rows by their exact outcome text (e.g. "True ASDS",
  "False Ocean").
- `ORDER BY count_outcomes DESC` ranks the outcomes from most to least frequent within that period.

## Close the Database Connection

In [12]:
conn.close()
print("Connection closed.")

Connection closed.


**Explanation:** `conn.close()` closes the database connection, freeing up the resources it
was using. This is good practice once all queries are finished, though since this was an in-memory
database, its contents are discarded regardless once the Colab session ends.


